In [77]:
import numpy as np
import os
import math
import random
from graphviz import Digraph

In [78]:
class Value:
    def __init__(self, data, _children=(), _op=() ):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')

        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out =  Value(self.data * other.data, (self,other), '*')

        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward

        return out
    def __pow__(self,exponent):
        assert isinstance(exponent, (int, float))
        out = Value(self.data**exponent, (self,), f"**{exponent}")

        def _backward():
            self.grad += exponent*(self.data**(exponent-1))*out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data / other.data, (self,other), '/')
        def _backward():
            self.grad += (1/other.data)*out.grad
            other.grad += -(self.data/other.data**2)*out.grad
        out._backward = _backward
        return out
    def __sub__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data - other.data, (self,other), '-')

        def _backward():
            self.grad += 1*out.grad
            other.grad += -1*out.grad
        out._backward = _backward
        return out


    def tanh(self):
        x  = self.data
        out = Value((math.exp(2*x)-1)/(math.exp(2*x)+1), (self,), 'tanh')

        def _backward():
            self.grad = (1-out.data**2)*out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return Value(other) + (-self)
    def __rtruediv__(self, other): return Value(other) * self**-1

    def backward(self):
        topo=[]
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()


In [79]:
#NEURAL NET

class Neuron:
    def __init__(self, nin, nonlin = True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons =  [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self,x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nout):
        sizes = [nin] + nout
        self.layers =[Layer(sizes[i], sizes[i+1], nonlin = (i != len(nout)-1)) for i in range(len(nout))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x)==1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]




In [80]:
#TRAINING LOOP

X = [[4.0, 3.0], [-2.0, -1.0], [1.0, -1.5], [-2.0, -2.0], [-3.0, -5.0], [4.0, -1.0], [7.0, 9.0], [-1.0,5.0]]
y = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0]

model = MLP(2, [4, 4, 1])

for step in range(100):

    #forward
    ypred = [model(x) for x in X]
    loss = sum(((yp - Value(yt))**2 for yp, yt in zip(ypred, y)), Value(0.0))

    #backward
    for p in model.parameters():
        p.grad = 0
    loss.backward()

    #gradient update
    for p in model.parameters():
        p.data -= 0.05*p.grad

    print(f"step{step:2d} Loss{loss.data:4f}")

step 0 Loss15.078351
step 1 Loss6.884614
step 2 Loss6.036906
step 3 Loss5.526509
step 4 Loss5.034934
step 5 Loss4.526527
step 6 Loss3.949636
step 7 Loss3.218078
step 8 Loss2.301281
step 9 Loss1.380370
step10 Loss0.702806
step11 Loss0.382379
step12 Loss0.953445
step13 Loss9.911830
step14 Loss19.043079
step15 Loss15.342067
step16 Loss5.496553
step17 Loss3.848999
step18 Loss3.320398
step19 Loss2.936402
step20 Loss2.594489
step21 Loss2.225859
step22 Loss1.773531
step23 Loss1.268872
step24 Loss0.809352
step25 Loss0.350958
step26 Loss0.140743
step27 Loss0.099277
step28 Loss0.103319
step29 Loss0.173520
step30 Loss0.418823
step31 Loss1.086427
step32 Loss2.365867
step33 Loss3.033628
step34 Loss2.871145
step35 Loss1.393405
step36 Loss0.819508
step37 Loss0.424063
step38 Loss0.284455
step39 Loss0.190776
step40 Loss0.152237
step41 Loss0.117772
step42 Loss0.101001
step43 Loss0.082229
step44 Loss0.071811
step45 Loss0.059508
step46 Loss0.052077
step47 Loss0.043538
step48 Loss0.038067
step49 Loss0.0320

In [81]:
print(ypred)

[Value(data=1.0061501672668465, grad=0.012300334533692947), Value(data=1.000951690698389, grad=0.0019033813967781477), Value(data=-0.9894053908112441, grad=0.021189218377511754), Value(data=1.0006207135498293, grad=0.0012414270996585763), Value(data=1.005081716101274, grad=0.010163432202547895), Value(data=-1.0088892258853772, grad=-0.017778451770754433), Value(data=0.9922947664896761, grad=-0.015410467020647856), Value(data=-0.9985311519786183, grad=0.0029376960427633936)]


In [82]:
## WITH reLU
#NEURAL NET

class Neuron:
    def __init__(self, nin, nonlin = True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons =  [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self,x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nout):
        sizes = [nin] + nout
        self.layers =[Layer(sizes[i], sizes[i+1], nonlin = (i != len(nout)-1)) for i in range(len(nout))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x)==1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]




In [83]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    nodes, edges = trace(root)

    for n in nodes:
        # A rectangular node for every Value — shows data and grad
        dot.node(
            name=str(id(n)),
            label="{ data: %.4f | grad: %.4f }" % (n.data, n.grad),
            shape='record'
        )
        if n._op:
            # A small circle node for the operation that created it
            op_id = str(id(n)) + n._op
            dot.node(name=op_id, label=n._op)
            dot.edge(op_id, str(id(n)))   # op → output value

    for n1, n2 in edges:
        # input value → op node
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [84]:
#TRAINING LOOP

X = [[4.0, 3.0], [-2.0, -1.0], [1.0, -1.5], [-2.0, -2.0], [-3.0, -5.0], [4.0, -1.0], [7.0, 9.0], [-1.0,5.0]]
y = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0]

model = MLP(2, [4, 4, 1])

for step in range(500):

    #forward
    ypred = [model(x) for x in X]
    loss = sum(((yp - Value(yt))**2 for yp, yt in zip(ypred, y)), Value(0.0))

    #backward
    for p in model.parameters():
        p.grad = 0
    loss.backward()

    #gradient update
    for p in model.parameters():
        p.data -= 0.01*p.grad

    print(f"step{step:2d} Loss{loss.data:4f}")

print(ypred)

    ### The loss values can differ very much depending on the random that can be chosen so running it multiple times may sometimes give a higher loss value with reLU hoever tanh gives a more consistent output.


step 0 Loss27.339876
step 1 Loss8.281417
step 2 Loss7.369246
step 3 Loss6.533498
step 4 Loss6.055941
step 5 Loss5.789343
step 6 Loss5.604679
step 7 Loss5.440227
step 8 Loss5.278985
step 9 Loss5.114298
step10 Loss4.941649
step11 Loss4.757595
step12 Loss4.559868
step13 Loss4.347725
step14 Loss4.124760
step15 Loss3.890652
step16 Loss3.653689
step17 Loss3.416763
step18 Loss3.191799
step19 Loss2.983527
step20 Loss2.788888
step21 Loss2.608340
step22 Loss2.430181
step23 Loss2.260255
step24 Loss2.104422
step25 Loss1.952027
step26 Loss1.809105
step27 Loss1.674292
step28 Loss1.545449
step29 Loss1.426749
step30 Loss1.303155
step31 Loss1.167501
step32 Loss1.062475
step33 Loss0.972764
step34 Loss0.897635
step35 Loss0.832490
step36 Loss0.776209
step37 Loss0.726348
step38 Loss0.681816
step39 Loss0.641715
step40 Loss0.605239
step41 Loss0.571783
step42 Loss0.540892
step43 Loss0.512739
step44 Loss0.486607
step45 Loss0.462123
step46 Loss0.438634
step47 Loss0.415482
step48 Loss0.393863
step49 Loss0.374390

In [85]:
dot = draw_dot(loss)
dot.render('my_graph', format='svg', cleanup=True)

'my_graph.svg'